In [18]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

**USING CLASSIFICATION METHOD**

In [19]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nHouse Type distribution')
        print(df['house_type'].value_counts())
        print('\nFurnishing distribution')
        print(df['furnishing'].value_counts())

        return df

In [20]:
#identifying features to use 
def identify_features(df):
    print("-" * 60)
    print("IDENTIFY FEATURES")
    print("-" * 60)

    #feature to drop
    feature_to_drop = [
        "title",
        "region",
        "region_name"
    ]

    #numerical columns
    num_features = [
        "property_size(sqm)",
        "bathrooms",
        "bedrooms"
    ]

    #category features
    cat_features = [
        "house_type",
        "furnishing",
        "region_parent_name",
        "is_boost"
    ]

    print("\nFeatures to drop:", feature_to_drop)
    print("\nNumerical features:", num_features)
    print("\nCategorical features:", cat_features)

    return feature_to_drop, num_features, cat_features

In [21]:
#preparing the data
def prepare_data(df, num_features, cat_features, feature_to_drop):
    print("-" * 60)
    print("PREPARING DATA")
    print("-" * 60)

    #drop unrelevant data
    df_model = df.drop(columns=feature_to_drop)

    #specify label
    #feature to use 
    x = df_model.drop("price", axis=1)

    #target to use
    y = df_model["price"]

    print("\nFeature Shape:", x.shape)
    print("\nTarget Shape:", y.shape)
    print("\nFeature columns list:", list(x.columns))

    return x, y, list(x.columns)

In [22]:
#creating pipeline
def create_preprocessing_pipeline(num_features, cat_features):
    print("-" * 60)
    print("CREATE PREPROCESSING PIPELINE")
    print("-" * 60)

    #num columns
    numerical_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    #category features
    category_pipeline = Pipeline([
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])

    #column preprocessing
    preprocessing = ColumnTransformer([
        ("num", numerical_pipeline, num_features),
        ("cat", category_pipeline, cat_features)
    ])

    return preprocessing

In [23]:
def split_data(x, y, test_size=0.2, random_state=42):
    print("-" * 60)
    print("SPLITTING THE DATA")
    print("-" * 60)

    # x and y train and test
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=random_state,
    )

    print("\nTraining set size:", x_train.shape[0])
    print("\nTest set size:", x_test.shape[0])

    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    print("\nTest set price range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))

    return x_train, x_test, y_train, y_test

**Model - Linear Regression**

In [24]:
def train_model(x_train, y_train, preprocessing):
    print('\n' + "=" * 60)
    print("TRAIN MODEL")
    print("=" * 60)

    model = Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", LinearRegression())
    ])
    
    model.fit(x_train, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    # Get feature names after preprocessing
    feature_names = model.named_steps[
        "preprocessing"
    ].get_feature_names_out()

    # Get coefficients from the regression model
    coefficients = model.named_steps[
        "regressor"
    ].coef_.ravel()

    for feature, coef in zip(feature_names, coefficients):
        print(f"{feature}: {coef:.2f}")

    print(
        f"\nModel Intercept: "
        f"{float(model.named_steps['regressor'].intercept_):.2f}"
    )

    return model

In [25]:
def evaluate_model(model, x_train, x_test, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("MODEL PERFROMANCE")
    print('\n' + "=" * 60)
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(model, x_train, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return metrics

**MODEL - Random Forest Regression**

In [26]:
def train_rm_model(x_train, y_train, preprocessing):
    print('\n' + "=" * 60)
    print("TRAIN MODEL - RANDOM FOREST REGRESSION")
    print("=" * 60)

    # Create the model pipeline
    rm_model = Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Train the model
    rm_model.fit(x_train, y_train)

    print("\nModel Trained Successfully")
    print("\nFeature Importances")

    # Get feature names after preprocessing
    feature_names = rm_model.named_steps[
        "preprocessing"
    ].get_feature_names_out()

    # Get feature importances from the trained model
    importances = rm_model.named_steps[
        "regressor"
    ].feature_importances_

    # Display feature importance
    for feature, importance in sorted(zip(feature_names, importances),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{feature}: {importance:.4f}")

    return rm_model

In [27]:
def evaluate_rm_model(rm_model, x_train, x_test, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE RM MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = rm_model.predict(x_train)
    y_test_pred = rm_model.predict(x_test)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("MODEL PERFROMANCE")
    print('\n' + "=" * 60)
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, x_train, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    rm_metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return rm_metrics

In [28]:
def save_model_artifact(rm_model, model, preprocessing, featured_columns):
    print('\n' + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_house_price_prediction.pkl")
    print("House Price Pediction for rm model saved successfully.")

    joblib.dump(model, "house_price_prediction.pkl")
    print("House Price Pediction model saved successfully.")

    joblib.dump(preprocessing, "preprocessing.pkl")
    print("Preprocessing saved successfully.")

    joblib.dump(featured_columns, "featured_columns.pkl")
    print("Featured columns saved successfully.")

In [29]:
def predict_car_price(year, make, model_name, condition, transmission):
    model = joblib.load("house_price_prediction.pkl")
    preprocessing = joblib.load("preprocessing.pkl")
    featured_columns = joblib.load("featured_columns.pkl")

    try:
        make_encoded = label_encoder["make"].transform([make])[0]
        model_encoded = label_encoder["model"].transform([model_name])[0]
        encoded_ordinals = ordinal_encoder.transform([[condition, transmission]])
        condition_encoded = encoded_ordinals[0, 0]
        transmission_encoded = encoded_ordinals[0, 1]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
        "year": year,
        "make_encoded": make_encoded,
        "model_encoded": model_encoded,
        "condition_encoded": condition_encoded,
        "transmission_encoded": transmission_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    feature_scale = scaler.transform(features)

    predicted_price = float(model.predict(feature_scale).ravel()[0])
    
    return predicted_price

In [30]:
def main():
    file_path = "jiji_housing_cleaned.csv"

    df = load_and_explore_dataset(file_path)

    feature_to_drop, num_features, cat_features = identify_features(df)

    x, y, featured_columns = prepare_data(df, num_features, cat_features, feature_to_drop)

    preprocessing = create_preprocessing_pipeline(num_features, cat_features)

    x_train, x_test, y_train, y_test = split_data(x, y)

    model = train_model(x_train, y_train, preprocessing)

    metrics = evaluate_model(model, x_train, x_test, y_train, y_test)

    rm_model = train_rm_model(x_train, y_train, preprocessing)

    rm_metrics = evaluate_rm_model(rm_model, x_train, x_test, y_train, y_test)

    save_model_artifact(rm_model, model, preprocessing, featured_columns)

    

In [31]:
if __name__ == "__main__":
    main()

------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(1238, 11)

check missing
title                 0
property_size(sqm)    0
house_type            0
bedrooms              0
bathrooms             0
furnishing            0
region                0
region_name           0
region_parent_name    0
is_boost              0
price                 0
dtype: int64

First five row
                                               title  property_size(sqm)  \
0         2bdrm Maisonette in Lekki Phase 1 for sale                 185   
1  4bdrm Duplex in Golden Park Estate, Sangotedo ...                 300   
2                       5bdrm Duplex in Idu for sale                 700   
3  Furnished 3bdrm Apartment in Kenneth Odidika, ...                 200   
4  3bdrm Townhouse/Terrace in Kaura Opposite for ...                 280   

          house_type  bedrooms  bathrooms      furnishing  

C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


MODEL PERFROMANCE


Training Set
 R2 score: 0.2980
 RMSE: 99539870.8727
 MAE: 82071652.7203

Testing Set
 R2 score: 0.3550
 RMSE: 93805098.1162
 MAE: 77404511.7208


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Cross Validation (5 Folds)
 R2 score: [0.26870512 0.3304705  0.17763908 0.25582594 0.13258251]
 Cross validation mean: 0.2330
 Cross validation std: 0.0699

TRAIN MODEL - RANDOM FOREST REGRESSION

Model Trained Successfully

Feature Importances
num__bedrooms: 0.2980
num__property_size(sqm): 0.1822
cat__house_type_Duplex: 0.1348
cat__region_parent_name_Lagos Island: 0.1256
num__bathrooms: 0.0587
cat__house_type_Bungalow: 0.0377
cat__furnishing_Unfurnished: 0.0341
cat__is_boost_Enterprise: 0.0282
cat__region_parent_name_Lagos Mainland: 0.0267
cat__house_type_Block of Flats: 0.0218
cat__furnishing_Semi-Furnished: 0.0183
cat__is_boost_False: 0.0156
cat__region_parent_name_Rivers State: 0.0050
cat__house_type_Maisonette: 0.0049
cat__region_parent_name_Oyo State: 0.0020
cat__house_type_Penthouse: 0.0019
cat__region_parent_name_Ogun State: 0.0012
cat__region_parent_name_Edo State: 0.0010
cat__is_boost_Vip_Gold: 0.0009
cat__house_type_Room & Parlour: 0.0004
cat__region_parent_name_Enugu State

...

MODEL PERFROMANCE


Training Set
 R2 score: 0.6575
 RMSE: 69527641.8073
 MAE: 54300760.7047

Testing Set
 R2 score: 0.4221
 RMSE: 88791772.3572
 MAE: 70173609.6261


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Cross Validation (5 Folds)
 R2 score: [0.44777336 0.5322783  0.39948271 0.37611398 0.36923861]
 Cross validation mean: 0.4250
 Cross validation std: 0.0603

SAVING MODEL ARTIFACT


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


NameError: name 'joblib' is not defined